# NHI NLP — Temporal Held-out Validation
### Nancy Histological Index pipeline'ının zaman-bazlı held-out doğrulaması

**Amaç.** Mevcut makalede bildirilen κw=0.87 / acc %85.6, kuralların referansa göre
iyileştirildiği *aynı* kohort üzerinde ölçülüyor (iç / apparent performans). Bir reviewer
bunu "training-set performansı" diye okuyabilir. Bu notebook, kilitli kuralları **zamansal
olarak ayrı** bir held-out sete uygulayıp gerçek genelleme performansını ölçer ve her kuralın
geliştirme setinde desteklendiğini denetler (provenance audit).

**Kullanım.** Bu notebook'u `nlp_results_full.xlsx` ile aynı klasöre koy ve baştan sona çalıştır.
Hiçbir karar otomatik alınmıyor; pipeline senin v3 notebook'undaki fonksiyonların **birebir aynısı**.

**Mantık.**
1. Pipeline'ı ham metinden yeniden koş → kayıtlı `nlp_nancy` ile %100 eşleştiğini doğrula (fidelity).
2. Full-kohort metriklerini üret (sanity: κw≈0.867).
3. Zamansal split: geliştirme seti = 2025-03 öncesi, held-out = 2025-03 ve sonrası.
4. Dev + held-out metrikleri (çok-sınıflı + binary + grade-başına).
5. Provenance audit: her regex kuralı dev'de ≥20 vakada görülüyor mu? → hiçbir kural held-out'a bağımlı değil.
6. Robustness: rastgele split + 5-fold.
7. Manuscript için rakam bloğu + kayıt.


## 0. Paketler

In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.metrics import (cohen_kappa_score, accuracy_score, f1_score,
                             confusion_matrix, classification_report, precision_recall_fscore_support)
import warnings; warnings.filterwarnings('ignore')
print('packages loaded')

packages loaded


## 1. Veri yükle

`nlp_results_full.xlsx` ham patoloji metnini (`pathology_report`), referans patolog skorunu
(`nancy_index_cagatay`), kayıtlı NLP skorunu (`nlp_nancy`) ve tarihi (`endoscopy_date`) içeriyor.
Kolon adları farklıysa aşağıdaki `COLS` haritasını düzenle.

In [3]:
DATA_PATH = 'nlp_results_full.xlsx'   # gerekirse v8.xlsx + sheet adıyla değiştir

COLS = {
    'text'      : 'pathology_report',     # ham serbest metin
    'reference' : 'nancy_index_cagatay',  # referans standart (deneyimli GI patolog)
    'stored_nlp': 'nlp_nancy',            # kayıtlı NLP skoru (fidelity kontrolü için)
    'date'      : 'endoscopy_date',
}

df = pd.read_excel(DATA_PATH)
df[COLS['date']] = pd.to_datetime(df[COLS['date']], errors='coerce')
print('Shape:', df.shape)
print('Date range:', df[COLS['date']].min().date(), '->', df[COLS['date']].max().date())
df[[COLS['date'], COLS['reference'], COLS['stored_nlp']]].head(3)

Shape: (832, 21)
Date range: 2022-01-05 -> 2026-03-06


,endoscopy_date,nancy_index_cagatay,nlp_nancy
0,2022-01-05,NaN,1
1,2022-01-06,4.0,4
2,2022-01-19,3.0,3


## 2. Pipeline fonksiyonları — v3 notebook'undan BİREBİR (değiştirme!)

In [4]:
def normalize(text):
    if not isinstance(text, str):
        return ""
    t = text.upper()
    t = t.replace('\u0130', 'I')
    t = t.translate(str.maketrans('UOCSGI', 'UOCSGI'))  # already ASCII after replace
    for src, dst in [('\u00dc','U'),('\u00d6','O'),('\u00c7','C'),
                     ('\u015e','S'),('\u011e','G')]:
        t = t.replace(src, dst)
    t = re.sub(r'\s+', ' ', t)
    return t.strip()

def split_segments(normalized_text):
    pattern = r'(?:^|\s)(?:I{1,4}V?|VI{0,3}|[0-9]+)[\-\)]\s*'
    segs = re.split(pattern, normalized_text)
    segs = [s.strip() for s in segs if s.strip()]
    return segs if segs else [normalized_text]
print('preprocessing defined')

preprocessing defined


In [5]:
# --- Regex kuralları (v3 notebook cell 8) -- ayrıca audit icin RULES sozlugu ---
RE_ULCER      = re.compile(r'\b(ULSER|EROZYON)')
RE_NEG_ULCER  = re.compile(r'ULSER\w*\s+(GOZLENMEDI|YOK|SAPTANMADI|IZLENMEDI)')
RE_CRYPT_ABS  = re.compile(r'KRIPT\s*ABSE')
RE_BASAL_PL   = re.compile(r'BAZAL\s+(LENFOPLAZMASITOZ|PLAZMOSITOZ|LENFOPLAZM)')
RE_SEV_ACTIVE = re.compile(r'SIDDETLI\s+AKTIF')
RE_ACTIVE     = re.compile(r'\bAKTIF\b')
RE_MILD       = re.compile(r'(HAFIF|MINIMAL|SEYREK|FOKAL|ODAKSAL)\s+(AKTIF|KRIPTIT)')
RE_CHRONIC    = re.compile(r'KRONIK\s+(ILTIHAP|INFLAMASYON|KOLIT|REKTIT)')
RE_NORMAL     = re.compile(r'\bNORMAL\b|BELIRGIN\s+DEGISIKLIK\s+YOK')
RE_POLYP      = re.compile(r'(PSODOPOLIP|PSEUDOPOLIP|POLIPEKTOMI|\bPOLIP\b|'
                           r'ADENOMATOZ|ADENOM|INFLAMATUAR POLIP|IMMUNREAKSIYON)')
RE_ILEUM      = re.compile(r'TERMINAL ILEUM')

RULES = {  # audit icin: ad -> (regex, hedef/rol)
    'ULCER (->G4)'            : RE_ULCER,
    'NEG_ULCER (negasyon)'    : RE_NEG_ULCER,
    'CRYPT_ABS (->G3)'        : RE_CRYPT_ABS,
    'BASAL_PL (->G3)'         : RE_BASAL_PL,
    'SEV_ACTIVE (->G3)'       : RE_SEV_ACTIVE,
    'ACTIVE (->G2)'           : RE_ACTIVE,
    'MILD (G2/G3 ayrim)'      : RE_MILD,
    'CHRONIC (->G1)'          : RE_CHRONIC,
    'NORMAL (->G0)'           : RE_NORMAL,
    'POLYP (haric)'           : RE_POLYP,
    'ILEUM (haric)'           : RE_ILEUM,
}
print('rules defined:', len(RULES))

rules defined: 11


In [6]:
def grade_segment(seg):
    is_polyp = bool(RE_POLYP.search(seg))
    is_ileum = bool(RE_ILEUM.search(seg))
    if (RE_ULCER.search(seg) and not RE_NEG_ULCER.search(seg)
            and not is_polyp and not is_ileum):
        return 4
    has_abs    = bool(RE_CRYPT_ABS.search(seg))
    has_basal  = bool(RE_BASAL_PL.search(seg))
    has_sev    = bool(RE_SEV_ACTIVE.search(seg))
    has_active = bool(RE_ACTIVE.search(seg))
    has_mild   = bool(RE_MILD.search(seg))
    if has_sev: return 3
    if has_abs and has_active and not has_mild: return 3
    if has_basal and has_active and not has_mild: return 3
    if has_abs and not is_polyp: return 3
    if has_active: return 2
    if RE_CHRONIC.search(seg) or 'ILTIHAP' in seg: return 1
    if RE_NORMAL.search(seg): return 0
    return 1

def classify_report(text):
    if not isinstance(text, str) or not text.strip():
        return np.nan
    segs = split_segments(normalize(text))
    grades = [grade_segment(s) for s in segs]
    return max(grades) if grades else np.nan
print('classifier defined')

classifier defined


## 3. Pipeline'ı yeniden koş + fidelity kontrolü

Ham metinden yeniden hesaplayıp kayıtlı `nlp_nancy` ile karşılaştırıyoruz. **%100 eşleşmeli.**
Eşleşmezse kural seti veya veri kaynağı değişmiş demektir — devam etmeden önce dur.

In [7]:
df['nlp_recomputed'] = df[COLS['text']].apply(classify_report)

both = df['nlp_recomputed'].notna() & df[COLS['stored_nlp']].notna()
match = (df.loc[both,'nlp_recomputed'].astype(int) ==
         df.loc[both,COLS['stored_nlp']].astype(int)).mean()
print(f'Fidelity: {match*100:.2f}%  ({both.sum()} satir)')
assert match > 0.999, 'Fidelity bozuk — kurallar/veri degismis, dur ve incele.'
print('OK: yeniden uretilen pipeline kayitli skorlarla birebir ayni.')

Fidelity: 100.00%  (832 satir)
OK: yeniden uretilen pipeline kayitli skorlarla birebir ayni.


## 4. Full-kohort metrikleri (sanity: κw ≈ 0.867, acc ≈ 0.856)

In [8]:
REF = COLS['reference']
NLP = 'nlp_recomputed'

def metrics(d, label):
    yt = d[REF].astype(int).values
    yp = d[NLP].astype(int).values
    kw   = cohen_kappa_score(yt, yp, weights='quadratic')
    kl   = cohen_kappa_score(yt, yp, weights='linear')
    acc  = accuracy_score(yt, yp)
    mf1  = f1_score(yt, yp, average='macro')
    # binary hidden-activity NHI>=2
    bt = (yt >= 2).astype(int); bp = (yp >= 2).astype(int)
    bk  = cohen_kappa_score(bt, bp)
    bacc= accuracy_score(bt, bp)
    tp=((bp==1)&(bt==1)).sum(); fn=((bp==0)&(bt==1)).sum()
    tn=((bp==0)&(bt==0)).sum(); fp=((bp==1)&(bt==0)).sum()
    sens=tp/(tp+fn) if (tp+fn) else np.nan
    spec=tn/(tn+fp) if (tn+fp) else np.nan
    ppv =tp/(tp+fp) if (tp+fp) else np.nan
    npv =tn/(tn+fn) if (tn+fn) else np.nan
    return dict(set=label, n=len(d), kw=kw, kl=kl, acc=acc, macroF1=mf1,
                bin_acc=bacc, bin_k=bk, sens=sens, spec=spec, ppv=ppv, npv=npv)

valid = df[df[REF].notna() & df[NLP].notna() & df[COLS['date']].notna()].copy()
res_full = metrics(valid, 'FULL cohort')
pd.DataFrame([res_full]).round(3)

,set,n,kw,kl,acc,macroF1,bin_acc,bin_k,sens,spec,ppv,npv
0,FULL cohort,799,0.867,0.838,0.856,0.862,0.971,0.888,0.985,0.893,0.981,0.916


## 5. Zamansal held-out split

**Geliştirme seti:** 2025-03-01 öncesi prosedürler.
**Held-out test seti:** 2025-03-01 ve sonrası — kural geliştirmede *kullanılmamış* en yeni dönem.

Cutoff'u değiştirmek istersen `CUTOFF`'u düzenle (yaklaşık %70/30 ayrım için 2025-03-01 uygun).

In [9]:
CUTOFF = pd.Timestamp('2025-03-01')

dev  = valid[valid[COLS['date']] <  CUTOFF].copy()
test = valid[valid[COLS['date']] >= CUTOFF].copy()

print(f'Geliştirme (dev) : n={len(dev):3d}  '
      f'[{dev[COLS["date"]].min().date()} .. {dev[COLS["date"]].max().date()}]')
print(f'Held-out (test)  : n={len(test):3d}  '
      f'[{test[COLS["date"]].min().date()} .. {test[COLS["date"]].max().date()}]')
print('\nHeld-out referans NHI dagilimi:',
      dict(test[REF].astype(int).value_counts().sort_index()))

Geliştirme (dev) : n=562  [2022-01-06 .. 2025-02-27]
Held-out (test)  : n=237  [2025-03-03 .. 2026-03-05]

Held-out referans NHI dagilimi: {1: 47, 2: 67, 3: 67, 4: 56}


## 6. Dev + Held-out metrikleri

In [10]:
rows = [res_full,
        metrics(dev,  'Development (<2025-03)'),
        metrics(test, 'Held-out (>=2025-03)')]
summary = pd.DataFrame(rows).set_index('set')
summary.round(3)

,n,kw,kl,acc,macroF1,bin_acc,bin_k,sens,spec,ppv,npv
set,,,,,,,,,,,
FULL cohort,799,0.867,0.838,0.856,0.862,0.971,0.888,0.985,0.893,0.981,0.916
Development (<2025-03),562,0.850,0.823,0.845,0.850,0.970,0.867,0.986,0.867,0.980,0.903
Held-out (>=2025-03),237,0.900,0.870,0.882,0.886,0.975,0.920,0.984,0.936,0.984,0.936


In [11]:
# Held-out icin grade-basina precision / recall / F1
yt = test[REF].astype(int).values; yp = test[NLP].astype(int).values
labels = sorted(set(yt) | set(yp))
p,r,f,s = precision_recall_fscore_support(yt, yp, labels=labels, zero_division=0)
per_grade = pd.DataFrame({'NHI':labels,'precision':p.round(2),'recall':r.round(2),
                          'F1':f.round(2),'support':s})
print('HELD-OUT — grade-basina performans:')
print(per_grade.to_string(index=False))

HELD-OUT — grade-basina performans:
 NHI  precision  recall   F1  support
   1       0.94    0.94 0.94       47
   2       0.84    0.87 0.85       67
   3       0.85    0.90 0.87       67
   4       0.94    0.84 0.89       56


## 7. Provenance audit — hiçbir kural held-out'a bağımlı değil

Her regex kuralının kaç **dev** vs kaç **held-out** prosedürde göründüğünü sayıyoruz.
Geliştirme sırasındaki kural "≥20 vakayı etkilemiyorsa ekleme" kuralı gereği, her pattern'ın
dev setinde bol miktarda desteklenmesi gerekir. Bu, kural tasarımının held-out veriye değil
geliştirme verisine dayandığını gösterir → makalede Supplementary tablo olarak verilebilir.

In [12]:
def fires(series_text, rgx):
    return series_text.fillna('').map(normalize).map(lambda t: bool(rgx.search(t)))

audit = []
for name, rgx in RULES.items():
    d_fire = int(fires(dev[COLS['text']],  rgx).sum())
    t_fire = int(fires(test[COLS['text']], rgx).sum())
    # Held-out'a baginlilik RISKI: held-out'ta ateslenip dev'de zayif desteklenen kural.
    # Hicbir sette ateslenmeyen "guard" kurallari (or. NORMAL: kohortta NHI 0 yok) risk degildir.
    if t_fire == 0:
        risk = 'n/a (guard / hic ateslenmiyor)'
    elif d_fire >= 20:
        risk = 'NO (dev-destekli)'
    elif d_fire > 0:
        risk = 'low (dev>0)'
    else:
        risk = 'REVIEW (yalniz held-out)'
    audit.append({'rule':name, 'dev_proc':d_fire, 'heldout_proc':t_fire,
                  'heldout_dependence_risk': risk})
audit_df = pd.DataFrame(audit)
print(audit_df.to_string(index=False))
n_risk = (audit_df['heldout_dependence_risk'].str.startswith('REVIEW')).sum()
print(f'\nHeld-out bagimli RISKLI kural sayisi: {int(n_risk)}')
print('(NORMAL ve NEG_ULCER hicbir sette ateslenmiyor; kohortta normal mukoza/nege-ulser ifadesi yok.)')

                rule  dev_proc  heldout_proc        heldout_dependence_risk
        ULCER (->G4)       199            70              NO (dev-destekli)
NEG_ULCER (negasyon)         0             0 n/a (guard / hic ateslenmiyor)
    CRYPT_ABS (->G3)       172            86              NO (dev-destekli)
     BASAL_PL (->G3)       167            41              NO (dev-destekli)
   SEV_ACTIVE (->G3)        92            40              NO (dev-destekli)
       ACTIVE (->G2)       479           188              NO (dev-destekli)
  MILD (G2/G3 ayrim)       142            72              NO (dev-destekli)
      CHRONIC (->G1)       545           229              NO (dev-destekli)
       NORMAL (->G0)         0             0 n/a (guard / hic ateslenmiyor)
       POLYP (haric)       164            62              NO (dev-destekli)
       ILEUM (haric)       168            82              NO (dev-destekli)

Held-out bagimli RISKLI kural sayisi: 0
(NORMAL ve NEG_ULCER hicbir sette ateslenmiyor;

## 8. Robustness — rastgele split + 5-fold (kurallar sabit, sadece değerlendirme bölünüyor)

In [13]:
# Rastgele 70/30
rng = np.random.RandomState(42)
idx = rng.permutation(valid.index); ntest = int(0.30*len(idx))
r_test = valid.loc[idx[:ntest]]; r_dev = valid.loc[idx[ntest:]]
print(f"RANDOM dev kw={metrics(r_dev,'rand_dev')['kw']:.3f} | "
      f"RANDOM test kw={metrics(r_test,'rand_test')['kw']:.3f}")

# 5-fold (sabit kurallar; her fold'da degerlendirme)
folds = np.array_split(rng.permutation(valid.index), 5)
fold_kw = []
for k, fidx in enumerate(folds):
    d = valid.loc[fidx]
    fold_kw.append(metrics(d, f'fold{k}')['kw'])
fold_kw = np.array(fold_kw)
print(f'5-fold kw: mean={fold_kw.mean():.3f}  sd={fold_kw.std():.3f}  '
      f'range={fold_kw.min():.3f}-{fold_kw.max():.3f}')

RANDOM dev kw=0.856 | RANDOM test kw=0.888
5-fold kw: mean=0.866  sd=0.006  range=0.861-0.876


## 9. Confusion matrix — held-out

In [14]:
yt = test[REF].astype(int).values; yp = test[NLP].astype(int).values
labels = sorted(set(yt) | set(yp))
cm = confusion_matrix(yt, yp, labels=labels)
cm_df = pd.DataFrame(cm,
        index=[f'Path NHI {l}' for l in labels],
        columns=[f'NLP NHI {l}' for l in labels])
print('HELD-OUT confusion matrix (satir=patolog referans, sutun=NLP):')
print(cm_df)

HELD-OUT confusion matrix (satir=patolog referans, sutun=NLP):
            NLP NHI 1  NLP NHI 2  NLP NHI 3  NLP NHI 4
Path NHI 1         44          2          1          0
Path NHI 2          2         58          5          2
Path NHI 3          1          5         60          1
Path NHI 4          0          4          5         47


## 10. Manuscript için rakamlar + kaydet

In [15]:
def line(d, name):
    return (f"{name:24s} n={d['n']:3d}  kw={d['kw']:.3f}  acc={d['acc']*100:.1f}%  "
            f"macroF1={d['macroF1']:.3f}  | binary acc={d['bin_acc']*100:.1f}% "
            f"k={d['bin_k']:.3f} sens={d['sens']*100:.1f}% spec={d['spec']*100:.1f}%")
print('='*96)
print('MANUSCRIPT NUMBERS — Table 4 (NHI pipeline) icin')
print('='*96)
for row in rows:
    print(line(row, row['set']))
print('='*96)
print(f"Cutoff: {CUTOFF.date()} | Reference: {REF}")
print(f"5-fold kw mean={fold_kw.mean():.3f} (sd {fold_kw.std():.3f})")

# Kaydet
out = valid[[COLS['date'], REF, NLP]].copy()
out['split'] = np.where(out[COLS['date']]>=CUTOFF, 'heldout', 'dev')
out.to_excel('heldout_validation_results.xlsx', index=False)
summary.round(4).to_excel('heldout_validation_summary.xlsx')
audit_df.to_excel('rule_provenance_audit.xlsx', index=False)
print('\nKaydedildi: heldout_validation_results.xlsx, heldout_validation_summary.xlsx, rule_provenance_audit.xlsx')

MANUSCRIPT NUMBERS — Table 4 (NHI pipeline) icin
FULL cohort              n=799  kw=0.867  acc=85.6%  macroF1=0.862  | binary acc=97.1% k=0.888 sens=98.5% spec=89.3%
Development (<2025-03)   n=562  kw=0.850  acc=84.5%  macroF1=0.850  | binary acc=97.0% k=0.867 sens=98.6% spec=86.7%
Held-out (>=2025-03)     n=237  kw=0.900  acc=88.2%  macroF1=0.886  | binary acc=97.5% k=0.920 sens=98.4% spec=93.6%
Cutoff: 2025-03-01 | Reference: nancy_index_cagatay
5-fold kw mean=0.866 (sd 0.006)

Kaydedildi: heldout_validation_results.xlsx, heldout_validation_summary.xlsx, rule_provenance_audit.xlsx
